# Does wage growth predict US inflation?

**Questions**
1. Does wage growth help predict core PCE inflation once energy prices, labor-market slack and inflation expectations are accounted for?
2. Do models that use wage growth forecast core PCE inflation better *out of sample* than simple benchmarks, and better than the same model without wages?

**Approach.** Lead/lag (Granger) tests, a Phillips-curve regression with HAC standard errors, and an expanding-window backtest with an explicit embargo so no model sees outcomes that were not yet known. Everything here is predictive; none of it identifies causal effects.

**Data.** Monthly FRED series (see `src/fetch.py`). Requires a free API key in the `FRED_API_KEY` environment variable. This notebook is re-run automatically by a GitHub Action when new data arrive, so the outputs and the conclusions at the end reflect the latest data.

In [ ]:
import os, sys
# make `src` importable whether the notebook is run from the repo root or from notebooks/
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from src import models, summary
from src.analysis import derive
from src.fetch import build_dataset

warnings.filterwarnings("ignore", category=UserWarning)
pd.options.display.float_format = "{:.3f}".format

HORIZON = 6   # forecast core PCE inflation over the next 6 months
df = derive(build_dataset())
print(df.index.min().date(), "to", df.index.max().date(), f"({len(df)} months)")
df[["core_pce", "wages", "energy_cpi", "unrate", "infl_exp"]].tail()

## 1. Data checks

Missing months matter. The 2025 federal shutdown left October 2025 out of the CPI and household-survey series, and it cannot be collected afterwards. Those months stay missing here: nothing is interpolated, and any feature that would need the missing month is dropped. Core PCE, the target, was not affected. The most recent month is usually incomplete because PCE is released later than CPI and jobs data.

In [ ]:
cols = ["core_pce", "wages", "energy_cpi", "unrate", "infl_exp"]
missing = df[cols].isna()
print("Months with a missing model input (count by series):")
print(missing.sum()[lambda s: s > 0])
print("\nWhich months:", [d.strftime("%Y-%m") for d in missing[missing.any(axis=1)].index])

frame = models.build_features(df, horizon=HORIZON)
print("\nUsable rows after building features:", len(frame.dropna(subset=models.FULL_FEATURES + ["target"])))
models.adf_table(frame, ["infl_1m", "wage_1m"])

## 2. Do wages lead inflation? (Granger tests)

Low p-values mean the first series helps predict the second beyond its own past. The reverse direction tests the "wage-price spiral" feedback. Series that fail a stationarity test are first-differenced first.

In [ ]:
models.granger_table(frame, maxlag=6)

## 3. Phillips-curve regression

Future core inflation regressed on current inflation, energy prices, unemployment, expectations and wage growth. Forecast windows overlap, so residuals are autocorrelated and the standard errors are HAC (Newey-West).

In [ ]:
table, reg_summary = models.phillips_regression(frame, horizon=HORIZON)
print(reg_summary)
table

In [ ]:
# Is the role of wage growth stable before and after 2021?
models.regime_comparison(frame, horizon=HORIZON)

## 4. Out-of-sample backtest

Models are refit every month on rows whose outcomes were already observed at that time (`train_mask`). Benchmarks: trailing 12-month inflation (naive), an autoregressive ridge model, ridge with energy/slack/expectations, ridge with wage growth added, and gradient boosting. Accuracy is compared with RMSE, MAE and a Diebold-Mariano test against the naive benchmark.

In [ ]:
res = models.run_all(df, horizon=HORIZON)
preds = res["predictions"]
for label, tbl in res["scores"].items():
    print(label)
    display(tbl)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(preds.index, preds["actual"], color="black", lw=2.5, label="Actual")
for col in preds.columns.drop("actual"):
    ax.plot(preds.index, preds[col], lw=1.2, label=col)
ax.set_title(f"Walk-forward forecasts: core PCE inflation over the next {HORIZON} months (annualized %)")
ax.set_ylabel("%"); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. What does wage growth add?

Beating or losing to the naive benchmark does not answer the second question. The direct test compares the ridge model that has wage terms with the smaller models it contains: the same model without wages, and a model using past inflation only. Because the models are nested, the usual Diebold-Mariano test is biased towards the smaller model; the Clark-West test corrects for that. It is one-sided: a small p-value means wages improve the forecasts by more than estimation noise alone would produce.

In [ ]:
res["wage_value"]

## 6. Is the answer the same at other horizons?

The backtest repeated for 3-, 6- and 12-month-ahead inflation (naive and ridge models only). Ratios below 1 mean the wage model had the lower RMSE.

In [ ]:
horizons = models.horizon_robustness(df)
horizons

## 7. Conclusions

Generated from the results above, so they always match the numbers. The wording is chosen by fixed thresholds (5% significance), not by hand.

In [ ]:
display(Markdown("\n".join(f"- {b}" for b in summary.results_bullets(res, horizons))))

## Limitations

- About 20 years of monthly data and overlapping forecast windows mean few independent observations, and the period since 2023 is short.
- The 2020-22 period is unusual and dominates error metrics, so results are also shown from 2023 onward.
- Several horizons, lags and baselines are tested, and the p-values are not corrected for multiple comparisons. One significant result among several is weak evidence.
- Granger tests and regressions show predictive relationships, not causal effects.
- The backtest uses today's revised data, not the figures available at each forecast date.
- Average hourly earnings are affected by composition changes (for example in spring 2020).
- October 2025 is missing from the CPI and household-survey series because of the federal shutdown.